# Dupin · Fase 1 — Exploración y entendimiento del fraude

**Objetivo: entendimiento, no producto.** Caracterizar PaySim para fundamentar las
decisiones de las Fases 2 (features) y 3 (evaluación). No se entrena nada serio
aquí; lo que se construye es *comprensión documentada* y una *lista de hipótesis
de features*.

### Pregunta del proyecto
> ¿Se puede predecir el fraude con información disponible **antes** de la
> transacción, cuánto cae el rendimiento con split **temporal** honesto vs.
> aleatorio, y cuánto del rendimiento ingenuo es **fuga de etiqueta** de las
> columnas de balance?

### Qué produce esta fase
1. Prevalencia y desbalance del fraude.
2. Cómo se distribuye el fraude (tipo, monto, tiempo, entidad).
3. **Demostración empírica de la fuga de etiqueta** en las columnas de balance.
4. Verificación de viabilidad del corte temporal.
5. Lista de hipótesis de features de comportamiento para la Fase 2.

## 0. Setup y carga desde GCS

In [ ]:
!pip -q install gcsfs pandas matplotlib seaborn scikit-learn

In [ ]:
from google.colab import auth
auth.authenticate_user()

PROJECT_ID = "dupin-dupin"
BUCKET_RAW = "dupin-dupin-raw"
RAW_OBJECT = "raw/paysim/PS_20174392719_1491204439457_log.csv"
GCS_URI    = f"gs://{BUCKET_RAW}/{RAW_OBJECT}"
RANDOM_SEED = 42

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

# dtypes canónicos — espejo de data/schema.py (única fuente de verdad del esquema).
DTYPES = {
    "step": "int32", "type": "category", "amount": "float64",
    "nameOrig": "string", "oldbalanceOrg": "float64", "newbalanceOrig": "float64",
    "nameDest": "string", "oldbalanceDest": "float64", "newbalanceDest": "float64",
    "isFraud": "int8", "isFlaggedFraud": "int8",
}

df = pd.read_csv(GCS_URI, dtype=DTYPES, storage_options={"project": PROJECT_ID})
print(f"Filas: {len(df):,}   Columnas: {df.shape[1]}")
print(f"Memoria: {df.memory_usage(deep=True).sum()/1e6:,.0f} MB")
df.head()

## 1. Prevalencia del fraude

Cuán raro es el fraude define toda la estrategia de métricas (Fase 3): con
desbalance extremo, *accuracy* y ROC-AUC engañan; la cabecera será
precision-recall.

In [ ]:
n = len(df)
n_fraud = int(df["isFraud"].sum())
print(f"Total:        {n:,}")
print(f"Fraude:       {n_fraud:,}  ({n_fraud/n:.4%})")
print(f"Legítimo:     {n - n_fraud:,}")
print(f"Desbalance:   1 fraude por cada {n // max(n_fraud,1):,} transacciones")
print(f"isFlaggedFraud=1: {int(df['isFlaggedFraud'].sum()):,}")

In [ ]:
# ¿Qué tan bueno es el flag heurístico nativo de PaySim (isFlaggedFraud)?
# Baseline trivial a batir, no una feature.
ct = pd.crosstab(df["isFraud"], df["isFlaggedFraud"], margins=True)
print(ct)
caught = ct.loc[1, 1] if 1 in ct.columns else 0
print(f"\nFraude que el flag nativo atrapa: {caught} / {n_fraud} ({caught/n_fraud:.2%})")

## 2. Fraude por tipo de operación — filtro estructural

PaySim documenta que el fraude solo ocurre en ciertos tipos de operación.
Verifiquémoslo: si es así, es un **filtro estructural** que reduce el espacio del
problema para la Fase 2.

In [ ]:
ct = pd.crosstab(df["type"], df["isFraud"])
ct.columns = ["legit", "fraud"]
ct["fraud_rate"] = ct["fraud"] / (ct["legit"] + ct["fraud"])
display(ct.sort_values("fraud_rate", ascending=False))

ax = ct["fraud_rate"].sort_values(ascending=False).plot(
    kind="bar", figsize=(8, 3.5), title="Tasa de fraude por tipo de operación", color="crimson"
)
ax.set_ylabel("P(fraude | tipo)")
plt.tight_layout(); plt.show()

> **A documentar tras ejecutar:** ¿en qué tipos se concentra el fraude (se espera
> TRANSFER y CASH_OUT)? Si el fraude es cero en PAYMENT/DEBIT/CASH_IN, las features
> de comportamiento de la Fase 2 pueden enfocarse en esos tipos.

## 3. Distribución de montos: fraude vs. legítimo

In [ ]:
display(df.groupby("isFraud")["amount"].describe().T)

fig, ax = plt.subplots(1, 2, figsize=(13, 4))
for k, g in df.groupby("isFraud"):
    ax[0].hist(np.log10(g["amount"] + 1), bins=60, alpha=0.5, density=True, label=f"isFraud={k}")
ax[0].set_title("log10(amount+1) — densidad por clase")
ax[0].set_xlabel("log10(amount+1)"); ax[0].legend()

sns.boxplot(data=df.sample(min(300_000, n), random_state=RANDOM_SEED),
            x="isFraud", y="amount", ax=ax[1])
ax[1].set_yscale("log"); ax[1].set_title("amount (escala log) por clase")
plt.tight_layout(); plt.show()

## 4. Estructura temporal

`step` es el eje temporal (1 step = 1 hora; 744 steps = 30 días). Es lo que hace
posible el split honesto. Verificamos densidad y si el fraude está repartido a lo
largo del tiempo (necesario para tener fraude a ambos lados de un corte).

In [ ]:
df["hour"] = (df["step"] % 24).astype("int16")
df["day"]  = (df["step"] // 24).astype("int16")

vol = df.groupby("step").size()
frd = df.groupby("step")["isFraud"].sum()

fig, ax = plt.subplots(2, 1, figsize=(13, 6), sharex=True)
ax[0].plot(vol.index, vol.values); ax[0].set_title("Volumen de transacciones por step")
ax[0].set_ylabel("conteo")
ax[1].plot(frd.index, frd.values, color="crimson"); ax[1].set_title("Conteo de fraude por step")
ax[1].set_xlabel("step (1 = 1 hora · 744 = 30 días)"); ax[1].set_ylabel("fraudes")
plt.tight_layout(); plt.show()

In [ ]:
# Patrón circadiano: ¿el fraude tiene firma horaria distinta del volumen legítimo?
fig, ax = plt.subplots(1, 2, figsize=(13, 3.5))
df.groupby("hour").size().plot(kind="bar", ax=ax[0], title="Volumen por hora del día")
df.groupby("hour")["isFraud"].mean().plot(kind="bar", ax=ax[1], color="crimson",
                                          title="Tasa de fraude por hora del día")
plt.tight_layout(); plt.show()

## 5. Entidades: ¿hay historia para construir comportamiento?

Las features de la Fase 2 acumulan comportamiento por entidad (`nameOrig`,
`nameDest`). Eso exige que las entidades **se repitan**. Aquí medimos la
reincidencia y el rol de los comercios (`nameDest` que empieza por `M`).

> **Caveat conocido de PaySim:** muchos originadores son casi de un solo uso. Si
> la reincidencia por `nameOrig` es baja, las features de *velocidad por
> originador* serán esparsas — hallazgo clave para calibrar la Fase 2.

In [ ]:
df["dest_is_merchant"] = df["nameDest"].str.startswith("M")
df["orig_is_merchant"] = df["nameOrig"].str.startswith("M")

print("Originadores que son comercio (esperado ~0):", int(df["orig_is_merchant"].sum()))
print("Destinos que son comercio:", int(df["dest_is_merchant"].sum()), f"({df['dest_is_merchant'].mean():.1%})")

print(f"\nnameOrig únicos: {df['nameOrig'].nunique():,} / {n:,}")
print(f"nameDest únicos: {df['nameDest'].nunique():,} / {n:,}")

print("\nReincidencia de originadores (cuántas tx por nameOrig):")
print(df["nameOrig"].value_counts().describe())

print("\nFraude segun destino comercio vs no-comercio:")
display(pd.crosstab(df["dest_is_merchant"], df["isFraud"], normalize="index"))

## 6. ⚠️ La trampa de la fuga de etiqueta — el hallazgo central

PaySim **anula** las transacciones fraudulentas, por lo que las cuatro columnas de
balance (`oldbalanceOrg`, `newbalanceOrig`, `oldbalanceDest`, `newbalanceDest`)
codifican el etiquetado. Aquí lo **demostramos**: primero que separan el fraude
casi perfectamente, luego que un modelo ingenuo sobre ellas alcanza un AUC
absurdo. Ese número es **trampa**, no detección.

In [ ]:
d = df
err_orig = d["newbalanceOrig"] + d["amount"] - d["oldbalanceOrg"]
err_dest = d["oldbalanceDest"] + d["amount"] - d["newbalanceDest"]
d = d.assign(errBalanceOrig=err_orig, errBalanceDest=err_dest)

print("Medias de las columnas de balance por clase (mira el contraste fraude vs legit):")
display(d.groupby("isFraud")[["oldbalanceOrg", "newbalanceOrig", "oldbalanceDest",
                              "newbalanceDest", "errBalanceOrig", "errBalanceDest"]].mean().T)

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, average_precision_score

# DEMO DEL ERROR CLÁSICO: balance crudo + SPLIT ALEATORIO.
# Mezcla las DOS fugas a propósito (etiqueta + temporal). Es el número
# "optimista no desplegable" cuyo contraste con el honesto es el resultado.
leak_cols = ["oldbalanceOrg", "newbalanceOrig", "oldbalanceDest", "newbalanceDest",
             "errBalanceOrig", "errBalanceDest"]

samp = d.sample(n=min(800_000, len(d)), random_state=RANDOM_SEED)
X = StandardScaler().fit_transform(samp[leak_cols].fillna(0.0).to_numpy())
y = samp["isFraud"].to_numpy()
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3, random_state=RANDOM_SEED, stratify=y)

clf = LogisticRegression(max_iter=300, class_weight="balanced").fit(Xtr, ytr)
p = clf.predict_proba(Xte)[:, 1]
print(f"[INGENUO · balance crudo · split ALEATORIO]")
print(f"  ROC-AUC = {roc_auc_score(yte, p):.4f}")
print(f"  PR-AUC  = {average_precision_score(yte, p):.4f}")
print("\n^ Si esto es absurdamente alto, ES la fuga. No se despliega.")

> **Conclusión operativa (no negociable a partir de aquí):**
> las cuatro columnas de balance quedan **prohibidas como features crudas**. El
> número de arriba conflaciona fuga de etiqueta + fuga temporal; la **Fase 3**
> descompone el gap en sus dos causas y `leakage_audit.py` lo audita
> automáticamente. Las features honestas de la Fase 2 derivan de comportamiento
> causal, no de estas columnas.

## 7. Viabilidad del corte temporal

In [ ]:
CUT = 480  # día 20 → primeros 20 días train, últimos 10 test (steps [480, 744))
for name, part in [("train  [0, 480)", df[df["step"] < CUT]),
                   ("test  [480, 744)", df[df["step"] >= CUT])]:
    f = int(part["isFraud"].sum())
    print(f"{name}:  filas={len(part):>10,}   fraude={f:>6,}  ({f/len(part):.4%})")

print("\nViable si hay fraude suficiente a AMBOS lados (cientos+ por lado).")

## 8. Conclusiones e hipótesis de features para la Fase 2

> *Completar con los números reales tras ejecutar.* Plantilla de cierre:

**Entendimiento del dataset**
- Prevalencia de fraude: ____% (1 por cada ____ tx).
- El fraude se concentra en los tipos: ____.
- Firma temporal: ____. Corte temporal en step 480 viable: sí/no.
- Reincidencia de entidades: originadores casi de un solo uso → impacto en
  velocidad por `nameOrig`: ____.

**Columnas prohibidas (fuga de etiqueta confirmada)**
- `oldbalanceOrg`, `newbalanceOrig`, `oldbalanceDest`, `newbalanceDest` — crudas.

**Hipótesis de features de comportamiento (Fase 2)** — todas causales, calculadas
solo con información **anterior** al instante de la transacción:
1. **Razón de monto:** `amount` / promedio histórico de la entidad.
2. **Recencia:** tiempo (steps) desde la última transacción de la entidad.
3. **Velocidad:** conteo de transacciones de la entidad en ventanas (corto/medio plazo).
4. **Novedad del destino:** ¿primera vez que `nameOrig` paga a este `nameDest`?
5. **Desviación de monto:** z-score del monto vs. historia de la entidad.
6. **Tipo + comportamiento:** señal de tipo (TRANSFER/CASH_OUT) cruzada con desviación.
7. **(Derivada de balance, honesta):** errores de balance *solo si* se demuestra que
   no filtran etiqueta bajo auditoría — por defecto, excluidas en v1.

Estas hipótesis pasan a `features/config.py` (`FeatureConfig`: ventanas, entidades,
defaults) y `features/build_features.py` en la Fase 2.